In [1]:
%pip install rasterio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import rasterio
from rasterio.enums import Resampling

def aumentar_resolucao_dem(input_path, output_path, upscale_factor=3):
    """
    Aumenta a resolução de um arquivo DEM raster.

    Args:
        input_path (str): Caminho para o DEM de entrada.
        output_path (str): Caminho para salvar o DEM de saída.
        upscale_factor (int): Fator de aumento. Para ir de 30m para 10m, o fator é 3.
    """
    try:
        with rasterio.open(input_path) as src:
            profile = src.profile
            
            print(f"Arquivo original lido: '{input_path}'")
            print(f"Resolução original: {src.res[0]}m x {src.res[1]}m")
            print("-" * 30)

            # Calcula a nova transformação e dimensões
            # A nova transformação divide o tamanho do pixel pelo fator de aumento
            new_transform = src.transform * src.transform.scale(
                (1 / upscale_factor),
                (1 / upscale_factor)
            )

            # Atualiza os metadados para o novo arquivo
            profile.update({
                'transform': new_transform,
                'width': src.width * upscale_factor,
                'height': src.height * upscale_factor
            })

            # Cria o novo arquivo e escreve os dados reamostrados
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(
                    src.read(
                        out_shape=(src.count, profile['height'], profile['width']),
                        # Método de reamostragem CUBIC é ideal para dados de elevação
                        resampling=Resampling.cubic 
                    )
                )
            
            print(f"Resolução do DEM aumentada com sucesso!")
            print(f"Novo arquivo salvo em: '{output_path}'")
            print(f"Nova resolução: {profile['transform'][0]}m x {abs(profile['transform'][4])}m")

    except FileNotFoundError:
        print(f"ERRO: O arquivo de entrada '{input_path}' não foi encontrado.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")

# --- EXECUTE AQUI ---

# 1. Altere esta variável para o nome do seu arquivo DEM de 30 metros
input_dem_path = r"SEU_CAMINHO\fazenda_A.tif"

# 2. Defina o nome do arquivo de saída com a nova resolução
output_dem_path = r"SEU_CAMINHO\fazenda_A.tif"

# 3. Fator de aumento (para ir de 30m para 10m, o fator é 3)
factor = 3

# 4. Chama a função para executar o processo
aumentar_resolucao_dem(input_dem_path, output_dem_path, upscale_factor=factor)

Arquivo original lido: 'C:\Users\CLIENTE\Documents\Empresa\clientes\topografia\faz_thiago.tif'
Resolução original: 0.0002694945839793305m x 0.0002694945850340138m
------------------------------
Resolução do DEM aumentada com sucesso!
Novo arquivo salvo em: 'C:\Users\CLIENTE\Documents\Empresa\clientes\topografia\faz_thiago10m.tif'
Nova resolução: 8.983152799311016e-05m x 8.983152834467126e-05m
